# Recherche parallèle : la convention décalée d'un mois

*Recherche parallèle, descriptive, **sans modèle** : elle ne fait pas partie de la correction de la codification 1. Journal : [`tableau_resultats.md`](tableau_resultats.md).*

**Question** : à quel paiement correspond une codification ? Deux conventions sont comparées, pour la codification `PAY_n` du mois M :
- **A, convention actuelle du projet** : `PAY_n` porte sur le règlement de la facture `BILL_AMT(n+1)` (solde de fin du mois précédent) par le paiement `PAY_AMTn` (paiement du mois M) ;
- **B, convention décalée d'un mois** : `PAY_n` porte sur le règlement de la facture `BILL_AMTn` (solde de fin du mois M) par le paiement `PAY_AMT(n-1)` (paiement du mois suivant).

Le notebook `04` (section 5 bis) a montré que les nouveaux retards et les sorties du retard s'alignent sur le paiement du mois suivant. Ici, **tous les passages et toutes les correspondances** sont vérifiés sous les deux conventions : les correspondances entre codification et paiement augmentent-elles ou diminuent-elles avec la convention B ?

**Données** : niveau 0 (`data/creditcard_pret_ingestion.csv`), sans `dpnm`. Les clients du test du machine learning de septembre (même découpage que les notebooks 01 à 01f) sont écartés, pour ne pas regarder leur `PAY_1`. Les deux conventions ne se calculent ensemble que pour **mai à août** (`PAY_5` à `PAY_2`) : la convention A demande `BILL_AMT(n+1)`, absent pour avril ; la convention B demande `PAY_AMT(n-1)`, absent pour septembre. Chaque exemple est un client à un mois M ; un client compte jusqu'à quatre fois (une section compte aussi les clients distincts).

**Catégories de paiement** (part de la facture réglée par le paiement, comme `ratio_PAY_BILLn` dans `docs/colonnes_creees.md`) : **aucune facture due** (facture nulle ou négative) ; **rien payé** ; **moins de 3 %** (sous la mensualité minimale, seuil de `TYPE_USAGE`) ; **de 3 à moins de 100 %** (paiement partiel) ; **facture soldée** (100 % et plus).

## 1. Données et exemples

In [23]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import Markdown, display
from sklearn.model_selection import train_test_split

current_dir = Path.cwd()
root_dir = next(p for p in [current_dir] + list(current_dir.parents) if (p / "data").exists())
df = pd.read_csv(root_dir / "data" / "creditcard_pret_ingestion.csv").drop(columns="dpnm")   # dpnm jamais utilisé
PAY = [f"PAY_{n}" for n in range(1, 7)]

# Clients du test du ML de septembre (même découpage que les notebooks 01 à 01f) : écartés
connus = df[df["PAY_1"] != 1]
_, test_ml = train_test_split(connus.index, test_size=0.2, stratify=connus["PAY_1"], random_state=42)
df = df.drop(index=test_ml)
print(f"Clients retenus : {len(df)} (test du ML de septembre écarté : {len(test_ml)})")

MOIS = {2: "août", 3: "juillet", 4: "juin", 5: "mai"}
CATEGORIES = ["aucune facture due", "rien payé", "moins de 3 %", "de 3 à moins de 100 %", "facture soldée"]

def categorie(paiement, facture):
    ratio = paiement / facture.where(facture > 0) * 100
    return pd.Series(np.select([facture <= 0, paiement <= 0, ratio < 3, ratio < 100], CATEGORIES[:4], CATEGORIES[4]), index=paiement.index)

def famille(c):
    return c.where(c < 2, 2)

lignes = []
for n in range(2, 6):
    bloc = pd.DataFrame({
        "client": df.index, "mois": n,
        "codif": df[f"PAY_{n}"], "codif_avant": df[f"PAY_{n + 1}"],
        "codif_avant2": df[f"PAY_{n + 2}"] if n + 2 <= 6 else np.nan,
        "facture_A": df[f"BILL_AMT{n + 1}"], "paiement_A": df[f"PAY_AMT{n}"],
        "facture_B": df[f"BILL_AMT{n}"], "paiement_B": df[f"PAY_AMT{n - 1}"],
    }, index=df.index)
    lignes.append(bloc)
ex = pd.concat(lignes, ignore_index=True)
ex = ex[(ex["codif"] != 1) & (ex["codif_avant"] != 1)].reset_index(drop=True)   # codifications 1 (rares avant septembre) écartées
ex["categorie_A"] = categorie(ex["paiement_A"], ex["facture_A"])
ex["categorie_B"] = categorie(ex["paiement_B"], ex["facture_B"])
print(f"Exemples (client × mois, mai à août) : {len(ex)} ; clients distincts : {ex['client'].nunique()}")

Clients retenus : 24737 (test du ML de septembre écarté : 5263)
Exemples (client × mois, mai à août) : 98914 ; clients distincts : 24737


## 2. Les passages entre le mois précédent et M

Chaque exemple est rangé selon le passage de sa codification entre le mois précédent et M :
- **resté sain, même codification** et **resté sain, codification changée** (par exemple 0 → -1) ;
- **nouveau retard** (sain → 2 et plus) ;
- **sortie du retard** vers -2, vers -1, vers 0, en distinguant le retard **récent** (un seul mois de retard avant M) du retard **ancien** (deux mois ou plus) ;
- **retard maintenu au même niveau**, **retard aggravé** (un cran de plus), **retard qui baisse** (encore en retard, mais moins), récent ou ancien.

In [24]:
f, fa = famille(ex["codif"]), famille(ex["codif_avant"])
ancien = (ex["codif_avant"] >= 2) & (ex["codif_avant2"] >= 2)
anciennete = np.where(ancien, " (retard ancien)", " (retard récent)")
passage = np.select(
    [(fa != 2) & (f != 2) & (ex["codif"] == ex["codif_avant"]),
     (fa != 2) & (f != 2),
     (fa != 2) & (f == 2)],
    ["Resté sain, même codification", "Resté sain, codification changée", "Nouveau retard"], "")
passage = pd.Series(passage, index=ex.index)
sortie = (fa == 2) & (f != 2)
passage[sortie] = "Sortie vers " + ex.loc[sortie, "codif"].astype(int).astype(str) + pd.Series(anciennete, index=ex.index)[sortie]
reste = (fa == 2) & (f == 2)
type_retard = np.select([ex["codif"] == ex["codif_avant"], ex["codif"] > ex["codif_avant"]], ["Retard maintenu", "Retard aggravé"], "Retard qui baisse")
passage[reste] = pd.Series(type_retard, index=ex.index)[reste] + pd.Series(anciennete, index=ex.index)[reste]
ex["passage"] = passage
ORDRE = ["Resté sain, même codification", "Resté sain, codification changée", "Nouveau retard"] + \
        [f"Sortie vers {c} (retard {a})" for c in (-2, -1, 0) for a in ("récent", "ancien")] + \
        [f"{t} (retard {a})" for t in ("Retard maintenu", "Retard aggravé", "Retard qui baisse") for a in ("récent", "ancien")]
display(ex["passage"].value_counts().reindex(ORDRE).fillna(0).astype(int).rename("Exemples").to_frame())

,Exemples
passage,
"Resté sain, même codification",73725
"Resté sain, codification changée",8952
Nouveau retard,4545
Sortie vers -2 (retard récent),1
Sortie vers -2 (retard ancien),4
Sortie vers -1 (retard récent),624
Sortie vers -1 (retard ancien),200
Sortie vers 0 (retard récent),1665
Sortie vers 0 (retard ancien),746


**Comment lire** : le nombre d'exemples de chaque passage (mai à août réunis). Les passages à quelques dizaines d'exemples ne donnent qu'une tendance.

## 3. Le paiement de chaque passage, sous les deux conventions

Pour chaque passage, la répartition des catégories de paiement sous la convention A (paiement du mois M sur la facture du mois précédent) et sous la convention B (paiement du mois suivant sur la facture du mois M), en % des exemples du passage.

In [25]:
tables = []
for convention in ("A", "B"):
    t = pd.crosstab(ex["passage"], ex[f"categorie_{convention}"], normalize="index").reindex(index=ORDRE, columns=CATEGORIES).fillna(0) * 100
    t.insert(0, "Convention", convention)
    tables.append(t.round(1).reset_index().set_index(["passage", "Convention"]))
repartition = pd.concat(tables).reindex([(p, c) for p in ORDRE for c in ("A", "B")])
display(repartition)

aucune facture due  rien payé  \
passage                           Convention                                  
Resté sain, même codification     A                         15.6        1.1   
                                  B                         12.3        1.0   
Resté sain, codification changée  A                         21.4        7.9   
                                  B                         37.1        5.2   
Nouveau retard                    A                          4.3        3.6   
                                  B                          6.4       73.2   
Sortie vers -2 (retard récent)    A                          0.0        0.0   
                                  B                        100.0        0.0   
Sortie vers -2 (retard ancien)    A                          0.0       25.0   
                                  B                          0.0       25.0   
Sortie vers -1 (retard récent)    A                          0.0       83.7   
                                  B                          0.0        0.3   
Sortie vers -1 (retard ancien)    A                          0.0       79.0   
                                  B                          0.0        0.5   
Sortie vers 0 (retard récent)     A                          0.0       72.6   
                                  B                          0.0        0.1   
Sortie vers 0 (retard ancien)     A                          0.0       43.3   
                                  B                          0.0        0.0   
Retard maintenu (retard récent)   A                          8.7       51.1   
                                  B                          0.0       24.9   
Retard maintenu (retard ancien)   A                          0.0       26.0   
                                  B                          0.0       35.0   
Retard aggravé (retard récent)    A                          0.0       77.2   
                                  B                          0.0       89.3   
Retard aggravé (retard ancien)    A                          0.0       61.8   
                                  B                          0.0       86.2   
Retard qui baisse (retard récent) A                          0.0       72.3   
                                  B                          0.0        0.0   
Retard qui baisse (retard ancien) A                          0.0       79.5   
                                  B                          0.3       24.5   

                                              moins de 3 %  \
passage                           Convention                 
Resté sain, même codification     A                    3.5   
                                  B                    3.1   
Resté sain, codification changée  A                    3.6   
                                  B                    3.3   
Nouveau retard                    A                    3.5   
                                  B                   12.1   
Sortie vers -2 (retard récent)    A                    0.0   
                                  B                    0.0   
Sortie vers -2 (retard ancien)    A                    0.0   
                                  B                    0.0   
Sortie vers -1 (retard récent)    A                   11.9   
                                  B                    0.2   
Sortie vers -1 (retard ancien)    A                   10.5   
                                  B                    0.0   
Sortie vers 0 (retard récent)     A                   11.7   
                                  B                    0.4   
Sortie vers 0 (retard ancien)     A                    6.8   
                                  B                    0.0   
Retard maintenu (retard récent)   A                    7.0   
                                  B                    5.0   
Retard maintenu (retard ancien)   A                    3.3   
                                  B                    3.5   
Retard aggravé (retard récent

**Comment lire** : deux lignes par passage, A puis B ; chaque ligne fait 100 %. Une convention « colle » mieux quand chaque passage a un paiement cohérent avec lui : un nouveau retard plutôt avec « rien payé » ou « moins de 3 % », une sortie du retard plutôt avec un paiement réel, un client resté sain plutôt avec un paiement ou aucune facture due.

## 4. Les correspondances : augmentent-elles ou diminuent-elles ?

Chaque ligne est une situation **incohérente** (ou cohérente) entre la codification et le paiement ; sa part est calculée sous les deux conventions. La dernière colonne donne l'écart B - A en points.
- **Retard sur encours nul** : codification de 2 et plus alors qu'aucune facture n'est due (les « faux retards » du niveau 5 du nettoyage) ;
- **retard sur facture soldée** : codification de 2 et plus alors que la facture est payée en entier ;
- **nouveau retard malgré un paiement d'au moins 3 %** ;
- **sortie du retard sans rien payer** (facture due) ;
- **codification saine (-1 ou 0) sans rien payer** (facture due) ;
- **sens des codifications saines** (page 3.3 du site) : -1 avec la facture soldée (paiement au comptant), -2 sans facture due ou facture soldée, 0 avec un paiement partiel (crédit renouvelable).

In [26]:
retard, nouveau, sortie_m = ex["codif"] >= 2, ex["passage"] == "Nouveau retard", ex["passage"].str.startswith("Sortie")
sain_m1_0 = ex["codif"].isin([-1, 0])
SITUATIONS = {
    "Retard sur encours nul (parmi les retards)": (retard, lambda c: c == "aucune facture due"),
    "Retard sur facture soldée (parmi les retards)": (retard, lambda c: c == "facture soldée"),
    "Nouveau retard malgré un paiement de 3 % ou plus (parmi les nouveaux retards)": (nouveau, lambda c: c.isin(["de 3 à moins de 100 %", "facture soldée"])),
    "Sortie du retard sans rien payer (parmi les sorties)": (sortie_m, lambda c: c == "rien payé"),
    "Codification -1 ou 0 sans rien payer (parmi les -1 et 0)": (sain_m1_0, lambda c: c == "rien payé"),
    "-1 avec la facture soldée (parmi les -1)": (ex["codif"] == -1, lambda c: c == "facture soldée"),
    "-2 sans facture due ou facture soldée (parmi les -2)": (ex["codif"] == -2, lambda c: c.isin(["aucune facture due", "facture soldée"])),
    "0 avec un paiement partiel, 3 à 100 % (parmi les 0)": (ex["codif"] == 0, lambda c: c == "de 3 à moins de 100 %"),
}
lignes = []
for nom, (masque, test) in SITUATIONS.items():
    a = test(ex.loc[masque, "categorie_A"]).mean() * 100
    b = test(ex.loc[masque, "categorie_B"]).mean() * 100
    lignes.append({"Situation": nom, "Exemples": int(masque.sum()), "Convention A (%)": round(a, 1), "Convention B (%)": round(b, 1), "Écart B - A (points)": round(b - a, 1)})
display(pd.DataFrame(lignes).set_index("Situation"))

,Exemples,Convention A (%),Convention B (%),Écart B - A (points)
Situation,,,,
Retard sur encours nul (parmi les retards),12997,3.5,2.2,-1.3
Retard sur facture soldée (parmi les retards),12997,6.3,0.3,-6.0
Nouveau retard malgré un paiement de 3 % ou plus (parmi les nouveaux retards),4545,88.6,8.3,-80.4
Sortie du retard sans rien payer (parmi les sorties),3240,68.3,0.2,-68.1
Codification -1 ou 0 sans rien payer (parmi les -1 et 0),71696,5.2,1.7,-3.5
-1 avec la facture soldée (parmi les -1),19166,63.4,87.9,24.5
-2 sans facture due ou facture soldée (parmi les -2),14221,98.1,97.8,-0.3
"0 avec un paiement partiel, 3 à 100 % (parmi les 0)",52530,81.3,90.8,9.5


**Comment lire** : les cinq premières lignes sont des **incohérences** (plus la part est basse, mieux la convention colle) ; les trois dernières, le **sens attendu** des codifications saines (plus la part est haute, mieux elle colle). Un écart négatif sur les premières et positif sur les dernières va dans le sens de la convention B.

## 5. Mois par mois

Les principales situations, séparément pour chaque mois M (mai, juin, juillet, août), sous les deux conventions : le résultat doit tenir pour chaque mois, pas seulement en moyenne.

In [27]:
lignes = []
for n, nom_mois in MOIS.items():
    m = ex["mois"] == n
    for nom in ["Retard sur encours nul (parmi les retards)", "Nouveau retard malgré un paiement de 3 % ou plus (parmi les nouveaux retards)",
                "Sortie du retard sans rien payer (parmi les sorties)", "Codification -1 ou 0 sans rien payer (parmi les -1 et 0)"]:
        masque, test = SITUATIONS[nom]
        mm = masque & m
        lignes.append({"Mois M": nom_mois, "Situation": nom.split(" (")[0], "Exemples": int(mm.sum()),
                       "A (%)": round(test(ex.loc[mm, "categorie_A"]).mean() * 100, 1),
                       "B (%)": round(test(ex.loc[mm, "categorie_B"]).mean() * 100, 1)})
display(pd.DataFrame(lignes).set_index(["Situation", "Mois M"]).sort_index(level=0))

Exemples  A (%)  \
Situation                                        Mois M                     
Codification -1 ou 0 sans rien payer             août        17542    3.9   
                                                 juillet     17666    4.8   
                                                 juin        18088    6.2   
                                                 mai         18400    5.7   
Nouveau retard malgré un paiement de 3 % ou plus août         1355   95.3   
                                                 juillet      1376   84.2   
                                                 juin         1073   83.2   
                                                 mai           741   92.6   
Retard sur encours nul                           août         3895    3.9   
                                                 juillet      3599    4.3   
                                                 juin         2986    3.4   
                                                 mai          2517    2.1   
Sortie du retard sans rien payer                 août         1057   63.7   
                                                 juillet       762   66.3   
                                                 juin          603   72.5   
                                                 mai           818   73.0   

                                                          B (%)  
Situation                                        Mois M          
Codification -1 ou 0 sans rien payer             août       0.1  
                                                 juillet    0.1  
                                                 juin       2.3  
                                                 mai        4.1  
Nouveau retard malgré un paiement de 3 % ou plus août       7.7  
                                                 juillet   10.6  
                                                 juin       9.1  
                                                 mai        3.6  
Retard sur encours nul                           août       1.2  
                                                 juillet    3.1  
                                                 juin       2.8  
                                                 mai        1.9  
Sortie du retard sans rien payer                 août       0.2  
                                                 juillet    0.4  
                                                 juin       0.0  
                                                 mai        0.0

**Comment lire** : quatre lignes par situation, une par mois. Si l'écart entre A et B a le même sens tous les mois, le résultat ne vient pas d'un mois atypique.

## 6. En clients distincts

Un client compte jusqu'à quatre fois dans les exemples. Ici, chaque client compte une fois : nombre de clients qui ont **au moins une** incohérence (retard sur encours nul, nouveau retard malgré un paiement de 3 % ou plus, sortie du retard sans rien payer, codification -1 ou 0 sans rien payer) sur mai à août, sous chaque convention.

In [28]:
incoherences = list(SITUATIONS)[:5]
lignes = []
for convention in ("A", "B"):
    au_moins_une = pd.Series(False, index=ex.index)
    for nom in incoherences:
        masque, test = SITUATIONS[nom]
        au_moins_une |= masque & test(ex[f"categorie_{convention}"])
    clients = ex.loc[au_moins_une, "client"].nunique()
    lignes.append({"Convention": convention, "Clients avec au moins une incohérence": clients,
                   "Part des clients (%)": round(clients / ex["client"].nunique() * 100, 1),
                   "Exemples incohérents": int(au_moins_une.sum())})
display(pd.DataFrame(lignes).set_index("Convention"))

,Clients avec au moins une incohérence,Part des clients (%),Exemples incohérents
Convention,,,
A,5868,23.7,8207
B,1701,6.9,1864


**Comment lire** : une ligne par convention. Moins de clients concernés sous B veut dire que la convention décalée explique mieux les codifications client par client, et pas seulement en moyenne.

## 7. Septembre : les codifications 1 sous les deux conventions

Pour septembre, seule une partie de la convention B se calcule : la facture `BILL_AMT1` (solde de fin septembre) est connue, mais pas le paiement d'octobre qui la réglerait. On compare la part de codifications 1 selon la facture concernée : sous A, `BILL_AMT2` (solde de fin août) ; sous B, `BILL_AMT1`. Si la convention B est la bonne, les 1 devraient se concentrer sur les clients qui ont **une facture de septembre à régler** (`BILL_AMT1` > 0), dont l'issue dépend d'un paiement d'octobre que la banque ne connaît pas encore.

In [29]:
s = df.copy()
s["famille_aout"] = np.select([s["PAY_2"] >= 2, s["PAY_2"] == 1], ["retard en août", "1 en août"], "sain en août")
s["facture_A"] = np.where(s["BILL_AMT2"] > 0, "facture due (BILL_AMT2 > 0)", "aucune facture due (BILL_AMT2 <= 0)")
s["facture_B"] = np.where(s["BILL_AMT1"] > 0, "facture due (BILL_AMT1 > 0)", "aucune facture due (BILL_AMT1 <= 0)")
s["est_1"] = s["PAY_1"] == 1
for convention in ("A", "B"):
    t = s.groupby(["famille_aout", f"facture_{convention}"])["est_1"].agg(Clients="size", Codifiés_1="sum")
    t["Part de codifications 1 (%)"] = (t["Codifiés_1"] / t["Clients"] * 100).round(1)
    display(Markdown(f"**Convention {convention}** : facture {'de fin août (BILL_AMT2)' if convention == 'A' else 'de fin septembre (BILL_AMT1)'}"))
    display(t)

**Convention A** : facture de fin août (BILL_AMT2)

Clients  Codifiés_1  \
famille_aout   facture_A                                                  
1 en août      aucune facture due (BILL_AMT2 <= 0)       21          21   
               facture due (BILL_AMT2 > 0)                7           7   
retard en août aucune facture due (BILL_AMT2 <= 0)       47           0   
               facture due (BILL_AMT2 > 0)             3848        1824   
sain en août   aucune facture due (BILL_AMT2 <= 0)     2696        1154   
               facture due (BILL_AMT2 > 0)            18118         682   

                                                    Part de codifications 1 (%)  
famille_aout   facture_A                                                         
1 en août      aucune facture due (BILL_AMT2 <= 0)                        100.0  
               facture due (BILL_AMT2 > 0)                                100.0  
retard en août aucune facture due (BILL_AMT2 <= 0)                          0.0  
               facture due (BILL_AMT2 > 0)                                 47.4  
sain en août   aucune facture due (BILL_AMT2 <= 0)                         42.8  
               facture due (BILL_AMT2 > 0)                                  3.8

**Convention B** : facture de fin septembre (BILL_AMT1)

Clients  Codifiés_1  \
famille_aout   facture_B                                                  
1 en août      facture due (BILL_AMT1 > 0)               28          28   
retard en août aucune facture due (BILL_AMT1 <= 0)        1           1   
               facture due (BILL_AMT1 > 0)             3894        1823   
sain en août   aucune facture due (BILL_AMT1 <= 0)     2402        1688   
               facture due (BILL_AMT1 > 0)            18412         148   

                                                    Part de codifications 1 (%)  
famille_aout   facture_B                                                         
1 en août      facture due (BILL_AMT1 > 0)                                100.0  
retard en août aucune facture due (BILL_AMT1 <= 0)                        100.0  
               facture due (BILL_AMT1 > 0)                                 46.8  
sain en août   aucune facture due (BILL_AMT1 <= 0)                         70.3  
               facture due (BILL_AMT1 > 0)                                  0.8

**Comment lire** : pour chaque situation d'août et chaque état de la facture, la part des clients codifiés 1 en septembre. Sous la convention B, une part de 1 bien plus forte quand une facture de septembre est due (et presque nulle sinon) irait dans le sens d'un statut d'attente, posé en attendant le paiement d'octobre.

## 8. Conclusion

*À écrire par l'utilisateur après lecture des résultats.*